# Task 4 : Anomaly detection

In [6]:
import pandas as pd
from graph_modeling_functions import build_graph_from_data, generate_analysis_report

In [7]:
def detect_anomalies(node_report, betweenness_threshold=0.8, pagerank_threshold=0.2):
    """
    Identifies suspicious users based on topological heuristics.
    - Potential Raid Leader: High Betweenness + High Coreness + Low PageRank.
    - Bridge/Troll: High Betweenness + Linking different communities.
    """
    
    # Normalize metrics between 0 and 1 for easier heuristic comparison
    metrics = ['Betweenness_Score', 'PageRank_Score', 'Coreness_Level']
    node_report_norm = node_report.copy()
    for m in metrics:
        node_report_norm[m] = (node_report[m] - node_report[m].min()) / (node_report[m].max() - node_report[m].min())

    alerts = []

    for index, row in node_report_norm.iterrows():
        # HEURISTIC 1: The "Raid Leader" / "Bridge Troll"
        # High Betweenness (top 10%) but relatively low PageRank (not a community pillar)
        if row['Betweenness_Score'] > node_report_norm['Betweenness_Score'].quantile(betweenness_threshold):
            if row['PageRank_Score'] < node_report_norm['PageRank_Score'].quantile(pagerank_threshold):
                alerts.append({
                    'User': row['User'],
                    'Type': 'Potential Bridge/Troll',
                    'Reason': 'High influence as a bridge but low community authority (PageRank).',
                    'Severity': 'High'
                })

        # HEURISTIC 2: Coordinated Backbone Member
        # High Coreness but very low PageRank (could be a bot or "sleeper" account)
        if row['Coreness_Level'] == 1.0 and row['PageRank_Score'] < 0.05:
            alerts.append({
                'User': row['User'],
                'Type': 'Suspicious Core Member',
                'Reason': 'Deeply embedded in the network backbone but has zero organic influence.',
                'Severity': 'Medium'
            })

    return pd.DataFrame(alerts)

def alert_community_manager(alerts_df):
    """
    Formats the alerts for a human Community Manager or DSA review.
    """
    if alerts_df.empty:
        print("✅ No suspicious patterns detected in this snapshot.")
    else:
        print(f"🚨 ALERT: {len(alerts_df)} suspicious users flagged for DSA review.")
        print("-" * 30)
        print(alerts_df.sort_values(by='Severity'))

In [8]:
edges_file = 'graphs/all_edges_consolidated.csv'
posts_file = 'all_posts_active_subreddit.csv'

G, df_posts = build_graph_from_data(edges_file, posts_file)

# 1. Run the previous report function
node_report = generate_analysis_report(G, df_posts)

# 2. Run Anomaly Detection
alerts = detect_anomalies(node_report)

# 3. Output results
alert_community_manager(alerts)

Graph initialized: 49994 users, 151771 interactions.
Community Detection: Found 270 distinct groups.
Calculating Betweenness Centrality (using sampling for speed)...
Calculating PageRank Centrality...
🚨 ALERT: 31 suspicious users flagged for DSA review.
------------------------------
                    User                    Type  \
0    Reasonable_doubt_59  Suspicious Core Member   
28            pwrboredom  Suspicious Core Member   
27           Ok_Sea_6214  Suspicious Core Member   
26            beowulftoo  Suspicious Core Member   
25  Revenant_adinfinitum  Suspicious Core Member   
24  Delicious_Summer7839  Suspicious Core Member   
23              jsideris  Suspicious Core Member   
22         Savant_Guarde  Suspicious Core Member   
21               tensigh  Suspicious Core Member   
20  therealdocumentarian  Suspicious Core Member   
19       RealityCheck831  Suspicious Core Member   
18        user4517proton  Suspicious Core Member   
17         WolfieTooting  Suspicious Co